In [0]:
import requests as rq
import json as js
from pyspark.sql.functions import lit, current_timestamp

In [0]:
PER_PAGE = 200
BASE_URL = "https://api.openbrewerydb.org/v1/breweries"

LANDING_ZONE_PATH = "/Volumes/openbrew/brew/landing_zone"
BRONZE_PATH = "openbrew.brew_bronze.breweries"

In [0]:
def fetch_breweries():
    all_data = []
    page = 1
    
    print(f"Iniciando extração da API: {BASE_URL}")
    
    while True:
        try:
            response = rq.get(f"{BASE_URL}?page={page}&per_page={PER_PAGE}")
            response.raise_for_status()
            data = response.json()
            
            if not data:
                break
                
            all_data.extend(data)
            page += 1
            
        except rq.exceptions.RequestException as e:
            print(f"Erro na requisição: {e}")
            break
            
    return all_data

In [0]:
# 1. Extraindo dados brutos e salvando na camada landing zone
raw_data = fetch_breweries()
json_string = js.dumps(raw_data)

dbutils.fs.put(f"{LANDING_ZONE_PATH}/breweries_raw.json", json_string, overwrite=True)

In [0]:
# 2. Lendo o dados que acabou de salvar na lading zone
df_landing = spark.read.option("multiline", "true").json(f"{LANDING_ZONE_PATH}/breweries_raw.json")

In [0]:
# 3. Adicionando metadados para governança
df_bronze = df_landing.withColumn("ingestion_timestamp", current_timestamp()) \
                     .withColumn("source_file", lit(BASE_URL))

In [0]:
# 4. Salvando como tabela delta na camada bronze
df_bronze.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_PATH)

print(f"Sucesso! Tabela {BRONZE_PATH} no Unity Catalog.")

In [0]:
%sql
SELECT * FROM openbrew.brew_bronze.breweries LIMIT 10;